# scLUMEN on human pancreatic differentiation (PA)

In [1]:
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import torch

from scLUMEN.datasets import build_wfr_dataframe, load_dataset
from scLUMEN.inference import generate_trajectories
from scLUMEN.refine import refine_decoder
from scLUMEN.runner import (
    ExperimentConfig, evaluate, prepare_latent_data, resolve_device, save_predictions,
    set_seed, train_dynamics, train_latent_model,
)
from scLUMEN.vae import decode_latent

config = ExperimentConfig(
    dataset="pancreatic",
    data_dir="../data/pancreatic",
    output_dir="../results",
    seed=0,
)
device = resolve_device("cpu")
run_dir = config.run_dir
run_dir.mkdir(parents=True, exist_ok=True)

## Load the data

In [2]:
data = load_dataset(config.dataset, config.data_dir)
trajectory = data.trajectory
print("Time points:", data.n_timepoints, "| genes:", data.n_genes)
print("Cells per time point:", [len(cells) for cells in trajectory])
print("Training:", data.train_tps, "| held out:", data.test_tps)
train_frame = build_wfr_dataframe(trajectory, data.train_tps)

Time points: 8 | genes: 2000
Cells per time point: [6190, 6177, 3562, 12477, 6480, 6404, 6013, 3971]
Training: (0, 1, 3, 5, 7) | held out: (2, 4, 6)


## Stage 1: latent space

In [3]:
vae = train_latent_model(config, trajectory, train_frame, device)
torch.save(vae.state_dict(), run_dir / "vae.pt")

## Stage 2: unbalanced flow matching

In [4]:
set_seed(config.flow_seed)
latent_train, latent_x0 = prepare_latent_data(config, trajectory, train_frame, vae, device)
dynamics = train_dynamics(config, latent_train, device)
torch.save(dynamics.state_dict(), run_dir / "flow.pt")

## Inference

In [5]:
particles = generate_trajectories(latent_x0, dynamics, device, tuple(range(data.n_timepoints)))
latent, weights = particles["latent"], particles["mass"]
np.savez_compressed(run_dir / "particles.npz", **particles)
print("Latent particles:", latent.shape)

Latent particles: (8, 2000, 10)


## Evaluate the unrefined model

In [6]:
decode_all = lambda model: [np.asarray(decode_latent(model, latent[t], device=device)) for t in range(data.n_timepoints)]
unrefined = pd.DataFrame(evaluate(config, data, trajectory, decode_all(vae), weights, vae, device))
unrefined

,time_point,stage,prop_tv_unweighted,prop_tv_weighted,gene_corr_unweighted,gene_corr_weighted,ot_unweighted,ot_weighted,recon_floor
0,2,2,0.196698,0.145353,0.97302,0.976007,695.283078,680.123351,633.726979
1,4,4,0.177025,0.109798,0.97302,0.976007,678.511604,660.459183,620.152417
2,6,6,0.188762,0.076947,0.97302,0.976007,668.337690,648.268436,612.078672


## Stage 3: mass-aware decoder refinement

In [7]:
refine_decoder(
    vae,
    {t: trajectory[t] for t in data.train_tps},
    {t: latent[t] for t in data.train_tps},
    {t: weights[t] for t in data.train_tps},
    mode="both",
    steps=config.refine_steps,
    batch_size=config.refine_batch_size,
    lr=config.refine_lr,
    blur=config.refine_blur,
    seed=config.seed,
)
torch.save(vae.state_dict(), run_dir / "vae_refined.pt")

In [8]:
predictions = decode_all(vae)
refined = pd.DataFrame(evaluate(config, data, trajectory, predictions, weights, vae, device))
save_predictions(run_dir / "predictions.npz", data, predictions, weights)
metrics = pd.concat([unrefined.assign(decoder="none"), refined.assign(decoder="both")])
metrics.insert(0, "seed", config.seed)
metrics.insert(0, "dataset", config.dataset)
metrics.to_csv(run_dir / "metrics.csv", index=False)
refined

,time_point,stage,prop_tv_unweighted,prop_tv_weighted,gene_corr_unweighted,gene_corr_weighted,ot_unweighted,ot_weighted,recon_floor
0,2,2,0.186873,0.142728,0.972363,0.976758,686.162614,671.217205,637.893171
1,4,4,0.160025,0.090638,0.972363,0.976758,667.518372,650.772033,624.608025
2,6,6,0.172762,0.082844,0.972363,0.976758,656.325739,639.036778,615.580456


In [9]:
columns = [c for c in metrics if c.startswith(("ot_", "prop_", "gene_corr"))]
metrics.groupby("decoder")[columns].mean().round(3)

,prop_tv_unweighted,prop_tv_weighted,gene_corr_unweighted,gene_corr_weighted,ot_unweighted,ot_weighted
decoder,,,,,,
both,0.173,0.105,0.972,0.977,670.002,653.675
none,0.187,0.111,0.973,0.976,680.711,662.950
